In [ ]:
!pip install geopandas

In [ ]:
!pip install folium

In [ ]:
import pandas as pd
from pandas import Series, DataFrame
import numpy as np
import matplotlib.pyplot as plt 
import chardet

In [ ]:
import geopandas as gpd
from geopandas import GeoSeries
from shapely.geometry import Point, LineString
import folium 
from folium import Marker, GeoJson
from folium.plugins import MarkerCluster, HeatMap

## `1` Import data

Import the data located at [this link](https://github.com/alexanderquispe/Diplomado_PUCP/blob/main/_data/data_dengue_peru.csv). It has information on people infected with dengue at the district level for 2015 to 2021.

In [ ]:
data_dengue = pd.read_csv(r'../../_data/data_dengue_peru.csv')

In [ ]:
data_dengue.head(4)

Some cases in Ubigeo have 5 numbers instead of 6. This is due to an error when importing as that deletes the initial character for the cases that start with a 0. We will add a 0 at the beggining for those cases with less than 6 numbers. 

In [ ]:
data_dengue['Ubigeo'] = data_dengue['Ubigeo'].astype(str)

In [ ]:
data_dengue['Ubigeo'] = data_dengue['Ubigeo'].str.zfill(6)

In [ ]:
data_dengue.head(4)

## `2` Ubigeo for Departments and Provinces

Generate ubigeo for Departments and Provinces taking the first two and four numbers. Hint: Use [this code](https://stackoverflow.com/questions/35552874/get-first-letter-of-a-string-from-column).

In [ ]:
data_dengue['cod_dep'] = data_dengue['Ubigeo'].apply(lambda x: str(x)[:2])
data_dengue['cod_prov'] = data_dengue['Ubigeo'].apply(lambda x: str(x)[:4])

In [ ]:
data_dengue

In [ ]:
dist_shape = gpd.read_file( r'../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp')

In [ ]:
dist_shape.rename(columns={'UBIGEO': 'Ubigeo'}, inplace=True)
dist_shape.rename(columns={'NOMBDEP': 'Departamento'}, inplace=True)
dist_shape.rename(columns={'NOMBPROV': 'Provincia'}, inplace=True)
dist_shape.rename(columns={'NOMBDIST': 'Distrito'}, inplace=True)

In [ ]:
dist_shape = dist_shape[['Departamento','Provincia','Distrito','Ubigeo', 'geometry']]

In [ ]:
dist_shape

## `3` Number of cases in 2021 by district

Use geopandas to plot the number of cases in 2021 by the district using a continuous legend. Do not forget to indicate the color of NA values. Use [this shapefile](https://github.com/alexanderquispe/Diplomado_PUCP/tree/main/_data/LIMITE_DISTRITAL_2020_INEI).

In [ ]:
data_2021 = data_dengue.loc[data_dengue["Año"]== 2021]

In [ ]:
data_2021

Agrupar por distrito y sumar los casos


In [ ]:
data_2021['Casos'] = pd.to_numeric(data_2021['Casos'], errors='coerce').fillna(0)

In [ ]:
data_2021

In [ ]:
data_2021_agregado = data_2021.groupby(['Año','Ubigeo',  'Departamento', 'Provincia', 'Distrito'], as_index=False).agg({
    'Casos': 'sum',            # Sumamos los casos de dengue
    # 'geometry': 'first' no se incluye aquí porque se manejará en el paso siguiente
})

In [ ]:
data_2021_agregado

In [ ]:
data_shape21 = dist_shape.copy()

In [ ]:
data_shape21['Año'] = 2021

In [ ]:
data_shape21

In [ ]:
dataset21 = pd.merge(data_2021_agregado, data_shape21, how="right", on=["Ubigeo", "Departamento", "Provincia", "Distrito"])

In [ ]:
dataset21

In [ ]:
dataset21 = gpd.GeoDataFrame(dataset21, geometry='geometry')

In [ ]:
fig, ax = plt.subplots(figsize=(20, 20))

# Primero, dibuja todos los polígonos con un color base que representará los valores NaN.
# Puedes elegir un color que claramente indique que los datos están ausentes, como 'lightgrey'.
dataset21.plot(ax=ax, color='grey', edgecolor='black')

# Luego, dibuja encima solo aquellos polígonos que tienen un valor no-NaN en 'Casos'.
dataset21.dropna(subset=['Casos']).plot(column='Casos', cmap='Reds', 
                                         edgecolor='black', legend=True, ax=ax)


## `4` Number of cases in 2021 by province

Use geopandas to plot the number of cases in 2021 by the province using a continuous legend. Do not forget to indicate the color of NA values. Use [this shapefile](https://github.com/alexanderquispe/Diplomado_PUCP/tree/main/_data/LIMITE_DISTRITAL_2020_INEI). For this task, you will have to aggregate shapefiles at the province level.

4.1 First, we group by province and add up the cases.

In [ ]:
data_2021_province = data_2021.groupby(['Año','Ubigeo',  'Departamento', 'Provincia'], as_index=False).agg({
    'Casos': 'sum',            # We added the cases of dengue fever
    # 'geometry': 'first' no se incluye aquí porque se manejará en el paso siguiente
})

In [ ]:
data_2021_province

In [ ]:
data_shape_provinc = dist_shape.copy()

In [ ]:
data_shape_provinc

In [ ]:
geo_data_provincia = data_shape_provinc.dissolve(by='Provincia', aggfunc='first').reset_index()

In [ ]:
dataset21_province = pd.merge(data_2021_province, geo_data_provincia , how="right", on=["Ubigeo", "Departamento", "Provincia"])

In [ ]:
dataset21_province 

In [ ]:
dataset21_province = gpd.GeoDataFrame(dataset21_province, geometry='geometry')

In [ ]:
fig, ax = plt.subplots(figsize=(20, 20))

# Primero, dibuja todos los polígonos con un color base que representará los valores NaN.
# Puedes elegir un color que claramente indique que los datos están ausentes, como 'lightgrey'.
dataset21_province.plot(ax=ax, color='grey', edgecolor='black')

# Luego, dibuja encima solo aquellos polígonos que tienen un valor no-NaN en 'Casos'.
dataset21_province.dropna(subset=['Casos']).plot(column='Casos', cmap='Reds', 
                                         edgecolor='black', legend=True, ax=ax)


## `5` Number of cases in 2021 by department for all the years

Use geopandas to plot the number of cases in 2021 by the province using a continuous legend. Do not forget to indicate the color of NA values. Use [this shapefile](https://github.com/alexanderquispe/Diplomado_PUCP/tree/main/_data/LIMITE_DISTRITAL_2020_INEI). For this task, you will have to aggregate shapefiles at the province level.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

Primero, se formará una base de datos estandar para los 2015-2020.

In [ ]:
data_dengue["Año"].value_counts()

In [ ]:
data = {}
Year = np.arange(2016,2022)
for i in Year:
    data[f'{i}'] = data_dengue.loc[data_dengue["Año"]== i]
    data[f'{i}']['Casos'] = pd.to_numeric(data[f'{i}']['Casos'], errors='coerce').fillna(0)

Segundo, se combinan los datos del dengue con la data de departamentos que contienen la geometría

Con este fin, en primer lugar, se importa la base de datos con la geometría.

In [ ]:
dist_shape_dep = gpd.read_file( r'../../_data/INEI_LIMITE_DEPARTAMENTAL/INEI_LIMITE_DEPARTAMENTAL.shp')

In [ ]:
data_shape_department = dist_shape_dep.copy()

# This code allow us to recognize each column (Este codigo nos permite reconocer cada columna)
data_shape_department.rename(columns={'CCDD': 'Ubigeo'}, inplace=True)
data_shape_department.rename(columns={'NOMBDEP': 'Departamento'}, inplace=True)
data_shape_department[['Ubigeo', 'Departamento']]

Hacemos el "merge" para cada caso con Pandas.DataFrame.merge:

In [ ]:
data_department = {}
dataset = {}
dataset_dep = {}
dataset_dep_geo = {}

Year = np.arange(2016,2022)
for i in Year:
    data_department[f'{i}'] = data[f'{i}'].groupby(['Año','cod_dep',  'Departamento'], as_index=False).agg({
    'Casos': 'sum'
    })
    
    data_department[f'{i}'].rename(columns={'cod_dep':'Ubigeo'}, inplace=True)
    
    dataset[f'{i}'] = pd.merge(data_department[f'{i}'], data_shape_department, how="right", on=["Ubigeo", "Departamento"])
    dataset_dep[f'{i}'] = dataset[f'{i}'][['Año','Ubigeo', 'Departamento', 'Casos', 'geometry']]   
    dataset_dep_geo[f'{i}'] = gpd.GeoDataFrame(dataset_dep[f'{i}'], geometry='geometry')

In [ ]:
fig, axs = plt.subplots(2,3,figsize=(15, 10))
plt.suptitle("Casos de Dengue a nivel departamental")

for i,ax in zip(Year, axs.flat):
    dataset_dep_geo[f'{i}'].plot(ax=ax, color='grey', edgecolor='black')

    # Luego, dibuja encima solo aquellos polígonos que tienen un valor no-NaN en 'Casos'.
    dataset_dep_geo[f'{i}'].dropna(subset=['Casos']).plot(column='Casos', cmap='Reds', 
                                             edgecolor='black', legend=True, ax=ax)
    ax.set_title(f'{i}')
    

## `6` Number of cases in 2021 by department for all the years

Use geopandas to plot the number of cases by the department for all 2021 quarters using subplots. Every subplot for each quarter. Use a categorical legend with 5 bins. Do not forget to indicate the color of NA values. Use [this shapefile](https://github.com/alexanderquispe/Diplomado_PUCP/tree/main/_data/LIMITE_DISTRITAL_2020_INEI). For this task, you will have to aggregate shapefiles at the department level.

In [ ]:
geo_data_6 = dist_shape.copy()
geo_data_6 = geo_data_6.dissolve(by = 'Departamento', as_index = False)
data_dengue['Casos'] = pd.to_numeric(data_dengue['Casos'],errors='coerce').fillna(0)

In [ ]:
def cuartos(x):
    if (x >= 40):
        return "Q4"
    if (27 <= x < 40):
        return "Q3"
    if (14 <= x < 27):
        return "Q2"
    if (x < 14):
        return "Q1"
    return float ("nan")

In [ ]:
data_dengue['cuartos'] = data_dengue['Semana'].apply(lambda x: cuartos(x))

In [ ]:
preg6= data_dengue.groupby(['cuartos','Ubigeo'], as_index= False)[['Casos']].sum()

In [ ]:
preg6 = pd.merge(geo_data_6, preg6, how="left", left_on= "Ubigeo", right_on= "Ubigeo")

In [ ]:
bins1 = mapclassify.Quantiles(preg6[ 'Casos' ], k=5).bins
limites = pd.qcut(preg6[ 'Casos' ], 5, retbins = True,duplicates='drop' )[ 1 ][ 1: ]

In [ ]:
fig, axis = plt.subplots( nrows = 2, ncols = 2, figsize = ( 15, 15 ) )

cmap = 'coolwarm'

idx = 1

for i in range( 2 ):
    for j in range( 2 ):
        ax = axis[ i ][ j ]
        c = preg6.cuartos.unique()[ idx ] ##identifica los cuartiles, los 4

        plot_data = preg6[(( preg6[ 'cuartos' ] == c ) | (preg6[ 'cuartos' ].isna()))]
  
        plot_data.plot( column = 'Casos', 
              cmap = cmap, 
              ax =ax,         
              linestyle = '--', 
              edgecolor = 'black', 
              legend = True, 
              missing_kwds = dict( color = 'white' ), 
              scheme="User_Defined",

              classification_kwds=dict(bins =limites)
             )
    
        ax.set_title( c )
        
        idx = idx + 1